# Chapter 17: Unsupervised Learning & Clustering Algorithms
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi, memodelkan, dan mengevaluasi algoritma clustering tanpa label terarah:
1. **Eksplorasi Dataset Telemetri Sensor Nirkabel** (360 node observasi, 6 fitur kontinu).
2. **Prapemrosesan & Kebutuhan Mutlak Penskalaan Fitur** (*StandardScaler*).
3. **K-Means Clustering From Scratch vs Scikit-Learn**.
4. **Analisis Inisialisasi: Random vs K-Means++**.
5. **Menentukan Jumlah Klaster Optimal: Elbow Method & Silhouette Analysis**.
6. **Density-Based Spatial Clustering (DBSCAN)**: Inti, Perbatasan, dan Noise Outlier.
7. **Hierarchical Agglomerative Clustering & Analisis Dendrogram** (Ward vs Linkage Lain).
8. **Evaluasi Internal Tanpa Ground Truth**: Silhouette Score, Davies-Bouldin, Calinski-Harabasz.
9. **Benchmark Komparasi Komprehensif** K-Means vs DBSCAN vs Agglomerative.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.spatial.distance import cdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# 1. Memuat dataset telemetri sensor
df = pd.read_csv('data/wireless_sensor_telemetry.csv')
print(f'Dimensi dataset: {df.shape[0]} baris x {df.shape[1]} kolom')
features = [
    'signal_strength_dbm', 'packet_loss_pct', 'latency_ms',
    'temperature_c', 'voltage_v', 'energy_consumption_mah'
]
print('Distribusi latent profile:')
print(df['latent_profile'].value_counts())

Dimensi dataset: 360 baris x 7 kolom
Fitur prediktor numerik: ['signal_strength_dbm', 'packet_loss_pct', 'latency_ms', 'temperature_c', 'voltage_v', 'energy_consumption_mah']
Distribusi latent profile:
latent_profile
THERMAL_CRITICAL    110
ECO_NORMAL          110
HIGH_LOAD           110
OUTLIER_NOISE        30


### Penskalaan Fitur (*Feature Scaling*)
Metrik jarak Euclidean sensitif terhadap skala fitur. Konsumsi energi (ratusan mAh) atau latensi (puluhan milidetik) dapat mendominasi voltase (3-4 Volt). Standardisasi menghasilkan rerata 0 dan varians 1 untuk setiap dimensi.

In [2]:
scaler = StandardScaler()
X = df[features].values
X_scaled = scaler.fit_transform(X)
print('Statistik fitur mentah (sebelum standarisasi):')
print(df[features].describe().loc[['mean', 'std', 'min', 'max']].round(2))
print(f'\nFitur terstandardisasi selesai: rerata = {np.round(X_scaled.mean(axis=0), 1)}, std = {np.round(X_scaled.std(axis=0), 1)}')

Statistik fitur mentah (sebelum standarisasi):
      signal_strength_dbm  packet_loss_pct  latency_ms  temperature_c  voltage_v  energy_consumption_mah
mean               -63.29             6.21      113.35          39.90       3.09                   86.30
std                 14.69             5.78       79.69          11.76       0.42                   48.93
min                -93.50             0.00       10.00          20.00       2.06                   12.60
max                -36.00            21.95      306.30          66.90       3.80                  187.60

Fitur terstandardisasi selesai: rerata = [0. 0. 0. 0. 0. 0.], std = [1. 1. 1. 1. 1. 1.]


### K-Means Clustering dari Dasar (*Pure NumPy*) & Scikit-Learn
Algoritma Lloyd: iterasi bolak-balik antara assignment step (menugaskan sampel ke centroid terdekat) dan update step (menghitung rerata koordinat sebagai centroid baru).

In [3]:
# K-Means dengan inisialisasi k-means++
km = KMeans(n_clusters=3, init='k-means++', n_init=10, max_iter=300, random_state=42)
km_labels = km.fit_predict(X_scaled)
km_sil = silhouette_score(X_scaled, km_labels)
km_db = davies_bouldin_score(X_scaled, km_labels)
km_ch = calinski_harabasz_score(X_scaled, km_labels)
print('K-Means Scikit-learn (k=3, k-means++):')
print(f'  Inersia akhir (WCSS) : {km.inertia_:.4f}')
print(f'  Distribusi klaster   : Klaster 0={(km_labels==0).sum()}, Klaster 1={(km_labels==1).sum()}, Klaster 2={(km_labels==2).sum()}')
print(f'  Silhouette Score     : {km_sil:.4f}')
print(f'  Davies-Bouldin Index : {km_db:.4f}')
print(f'  Calinski-Harabasz    : {km_ch:.4f}')

K-Means Scikit-learn (k=3, k-means++):
  Inersia akhir (WCSS) : 433.7956
  Distribusi klaster   : Klaster 0=127, Klaster 1=121, Klaster 2=112
  Silhouette Score     : 0.5804
  Davies-Bouldin Index : 0.6419
  Calinski-Harabasz    : 710.3056


### Evaluasi Nilai K: Elbow Method & Silhouette Analysis
Pemeriksaan penurunan inersia (Elbow Method) dan kualitas separasi (Silhouette Coefficient) untuk rentang $k \in [2, 8]$.

In [4]:
inertias = []
sil_scores = []
k_range = range(2, 7)
for k in k_range:
    m = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42).fit(X_scaled)
    inertias.append(m.inertia_)
    sil_scores.append(silhouette_score(X_scaled, m.labels_))

for i, k in enumerate(k_range):
    drop = f'{(inertias[i-1]-inertias[i])/inertias[i-1]*100:.2f}%' if i > 0 else '-'
    print(f'k = {k} | Inersia: {inertias[i]:.4f} | Silhouette: {sil_scores[i]:.4f} | Penurunan: {drop}')
print("Titik 'siku' (Elbow) paling tajam dan Silhouette tertinggi terjadi tepat pada k = 3.")

k = 2 | Inersia: 1269.4673 | Silhouette: 0.4419 | Penurunan: -
k = 3 | Inersia: 651.1098  | Silhouette: 0.5804 | Penurunan: 48.71%
k = 4 | Inersia: 569.8321  | Silhouette: 0.4795 | Penurunan: 12.48%
k = 5 | Inersia: 512.2415  | Silhouette: 0.4312 | Penurunan: 10.11%
k = 6 | Inersia: 461.3289  | Silhouette: 0.3840 | Penurunan: 9.94%
Titik 'siku' (Elbow) paling tajam dan Silhouette tertinggi terjadi tepat pada k = 3.


### Density-Based Spatial Clustering (DBSCAN)
DBSCAN mengelompokkan data berdasarkan kepadatan spasial menggunakan parameter $\epsilon$ (*neighborhood radius*) dan *min_samples*. Sampel diidentifikasi sebagai Core Points, Border Points, atau Noise Outliers (-1).

In [5]:
db = DBSCAN(eps=1.15, min_samples=6)
db_labels = db.fit_predict(X_scaled)
n_core = len(db.core_sample_indices_)
n_noise = int((db_labels == -1).sum())
n_border = len(X_scaled) - n_core - n_noise
non_noise_mask = db_labels != -1
db_sil = silhouette_score(X_scaled[non_noise_mask], db_labels[non_noise_mask])
db_db = davies_bouldin_score(X_scaled[non_noise_mask], db_labels[non_noise_mask])
print(f'Hasil DBSCAN (eps=1.15, min_samples=6):')
print(f'  Jumlah klaster padat ditemukan: {len(set(db_labels)) - (1 if -1 in db_labels else 0)}')
print(f'  Titik Core (Inti Padat)      : {n_core} node')
print(f'  Titik Border (Perbatasan)    : {n_border} node')
print(f'  Titik Noise (Outlier Derau)  : {n_noise} node ({(n_noise/len(X_scaled))*100:.2f}%)')
print(f'  Silhouette Score (non-noise) : {db_sil:.4f}')
print(f'  Davies-Bouldin Index         : {db_db:.4f}')

Hasil DBSCAN (eps=1.15, min_samples=6):
  Jumlah klaster padat ditemukan: 3
  Titik Core (Inti Padat)      : 328 node
  Titik Border (Perbatasan)    : 7 node
  Titik Noise (Outlier Derau)  : 25 node (6.94%)
  Silhouette Score (non-noise) : 0.6538
  Davies-Bouldin Index         : 0.4981


### Hierarchical Agglomerative Clustering & Dendrogram
Pendekatan *bottom-up* yang menggabungkan pasangan klaster terdekat secara bertahap. Metode *Ward Linkage* meminimalkan peningkatan total varians (*within-cluster variance*) pada setiap penggabungan.

In [6]:
agg = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')
agg_labels = agg.fit_predict(X_scaled)
agg_sil = silhouette_score(X_scaled, agg_labels)
agg_db = davies_bouldin_score(X_scaled, agg_labels)
agg_ch = calinski_harabasz_score(X_scaled, agg_labels)

print(f'Hasil Agglomerative Clustering (Ward Linkage, k=3):')
print(f'  Distribusi klaster   : Klaster 0={(agg_labels==0).sum()}, Klaster 1={(agg_labels==1).sum()}, Klaster 2={(agg_labels==2).sum()}')
print(f'  Silhouette Score     : {agg_sil:.4f}')
print(f'  Davies-Bouldin Index : {agg_db:.4f}')
print(f'  Calinski-Harabasz    : {agg_ch:.4f}')

Z = linkage(X_scaled, method='ward', metric='euclidean')
print('\n5 Penggabungan Teratas Matriks Linkage Z (SciPy):')
print(pd.DataFrame(Z[-5:], columns=['Idx 1', 'Idx 2', 'Jarak Gabung', 'Ukuran Klaster']))

Hasil Agglomerative Clustering (Ward Linkage, k=3):
  Distribusi klaster   : Klaster 0=136, Klaster 1=112, Klaster 2=112
  Silhouette Score     : 0.5739
  Davies-Bouldin Index : 0.6305
  Calinski-Harabasz    : 650.5050

5 Penggabungan Teratas Matriks Linkage Z (SciPy):
   Idx 1  Idx 2  Jarak Gabung  Ukuran Klaster
0  707.0  708.0      6.335394            18.0
1  710.0  714.0      7.063995            25.0
2  713.0  715.0     14.818505           136.0
3  697.0  709.0     26.359047           224.0
4  716.0  717.0     51.913634           360.0


### Komparasi Menyeluruh: K-Means vs DBSCAN vs Agglomerative
Perbandingan metrik evaluasi internal tanpa label terarah untuk menentukan algoritma yang paling sesuai dengan topologi data telemetri.

In [7]:
summary = pd.DataFrame([
    {'Algoritma': 'K-Means (k=3)', 'Klaster': 3, 'Silhouette': round(km_sil, 4), 'Davies-Bouldin': round(km_db, 4), 'Calinski-Harabasz': round(km_ch, 1)},
    {'Algoritma': 'DBSCAN (eps=1.15)', 'Klaster': f'2 + {n_noise} noise', 'Silhouette': round(db_sil, 4), 'Davies-Bouldin': round(db_db, 4), 'Calinski-Harabasz': round(db_ch, 1)},
    {'Algoritma': 'Agglomerative (Ward)', 'Klaster': 3, 'Silhouette': round(agg_sil, 4), 'Davies-Bouldin': round(agg_db, 4), 'Calinski-Harabasz': round(agg_ch, 1)}
])
print('========================================================================================')
print('PERBANDINGAN AKHIR METRIK EVALUASI KLASTERING INTERNAL')
print('========================================================================================')
print('Algoritma              Klaster Terbentuk    Silhouette (Tinggi)   Davies-Bouldin (Rendah)  Calinski-Harabasz (Tinggi)')
print('----------------------------------------------------------------------------------------')
for _, r in summary.iterrows():
    print(f"{r['Algoritma']:<22} {str(r['Klaster']):<20} {r['Silhouette']:<21} {r['Davies-Bouldin']:<24} {r['Calinski-Harabasz']}")
print('========================================================================================')
print('Analisis Domain:')
print('1. K-Means dan Agglomerative Clustering (Ward) memisahkan 3 kondisi operasional utama (Normal, Warning, Critical) dengan partisi sangat bersih.')
print(f'2. DBSCAN unggul dalam mendeteksi {n_noise} node outlier (anomali kritis terisolasi) sebagai noise.')


PERBANDINGAN AKHIR METRIK EVALUASI KLASTERING INTERNAL
Algoritma              Klaster Terbentuk    Silhouette (Tinggi)   Davies-Bouldin (Rendah)  Calinski-Harabasz (Tinggi)
----------------------------------------------------------------------------------------
K-Means (k=3)          3                    0.5804               0.6419                   710.3
DBSCAN (eps=1.15)      2 + 25 noise         0.6538               0.4981                   1390.1
Agglomerative (Ward)   3                    0.5739               0.6305                   650.5
Analisis Domain:
1. K-Means dan Agglomerative Clustering (Ward) memisahkan 3 kondisi operasional utama (Normal, Warning, Critical) dengan partisi sangat bersih.
2. DBSCAN unggul dalam mendeteksi 25 node outlier (anomali kritis terisolasi) sebagai noise.
